# Week 3 — Pipeline PySpark: HuggingFace EUR-Lex → Bronze → Silver

**Objetivo de la sesión:** ingestar el dataset `NLP-AUEB/eurlex` desde HuggingFace, persistirlo en `data/bronze/` (raw) y transformarlo a `data/silver/` (limpio, estructurado, particionado).

**Por qué un notebook de exploración antes de escribir módulos:**
Hasta que no hemos visto la *forma real* de los datos (columnas, tipos, NULLs, longitudes...), cualquier decisión de schema es ficción. El notebook es donde **descubrimos** la realidad. Una vez la sabemos, extraemos el código a `spark/` (módulos reutilizables, testables, orquestables por Airflow).

**Recordatorio del medallion:**
- **Bronze** (Parte 1) = raw, fiel al origen. Mínima transformación: estructura los datos pero no los limpia.
- **Silver** (Parte 2) = limpio, normalizado, particionado por `year`/`doc_type`.
- **Gold** = entidades y relaciones listas para Neo4j.

## Bloque 0 — Setup

**Diferencias con Week 2:**
- Añadimos `datasets` (cliente HuggingFace) y `pyarrow` (Parquet writer).
- Subimos `spark.sql.shuffle.partitions` a 8: el dataset ya no es de 8 filas como en Week 2 — ahora son cientos/miles.
- Añadimos `spark.driver.memory` por si el job hace `toPandas` o `collect` durante exploración.

In [ ]:
import pyspark                                          # versión de PySpark del contenedor
from pyspark.sql import SparkSession                    # punto de entrada Spark
from pyspark.sql import functions as F                  # alias estándar para funciones
from pyspark.sql.types import (                          # tipos para definir StructType
    StructType, StructField, StringType, IntegerType, ArrayType, LongType,
)
import datasets as hf_ds                                # cliente HuggingFace datasets
import pyarrow                                          # backend de Parquet/Arrow
from pathlib import Path

print(f"pyspark   = {pyspark.__version__}")
print(f"datasets  = {hf_ds.__version__}")
print(f"pyarrow   = {pyarrow.__version__}")

In [ ]:
# SparkSession: la túnica que vamos a usar durante todo el notebook.
spark = (
    SparkSession.builder
    .appName("EUGraphRAG-Week3-Bronze")                  # visible en la Spark UI :4040
    .master("local[*]")                                  # local con todos los cores del contenedor
    .config("spark.sql.shuffle.partitions", "8")         # 8 vs default 200; dataset pequeño-mediano
    .config("spark.driver.memory", "2g")                 # margen para toPandas/collect en exploración
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}, cores: {spark.sparkContext.defaultParallelism}")

## Bloque 1 — Cargar `NLP-AUEB/eurlex` desde HuggingFace\n\n**Concepto nuevo: `datasets.load_dataset()`**. Es el equivalente HF de `spark.read.csv()` o `pd.read_csv()`. Bajo el capó:\n1. Descarga los archivos del Hub la primera vez (a `~/.cache/huggingface/`).\n2. Los expone como un `DatasetDict` (splits) o `Dataset` (un solo split).\n3. Internamente cada `Dataset` está respaldado por un fichero Arrow (memory-mapped), así que es **eficiente** y soporta filas grandes sin cargar todo en RAM.\n\n**Decisión: subset_size.** El dataset entero tiene ~57k docs. Para esta sesión de exploración nos quedamos con 500 — carga rápido y permite iterar sin esperar. En producción llamarías con `subset_size=None` para todo el dataset.\n\n**Sintaxis de slicing HF:** `split='train[:500]'` no descarga sólo 500 filas — descarga el split entero, pero solo carga las 500 primeras. Es lógico: HF empaqueta el dataset entero en archivos Parquet/Arrow, no expone descargas parciales del fichero.

In [ ]:
SUBSET_SIZE = 500                                       # para exploración; None = todo el dataset
SPLIT       = "train"                                   # 'train' / 'validation' / 'test'

split_expr = f"{SPLIT}[:{SUBSET_SIZE}]" if SUBSET_SIZE else SPLIT

# OJO: el ID del dataset en HF es CASE-SENSITIVE. El namespace correcto es
# "NLP-AUEB" en mayúsculas (no "nlpaueb"). Es uno de los errores más típicos
# al copiar nombres de la doc.
ds = hf_ds.load_dataset("NLP-AUEB/eurlex", split=split_expr, trust_remote_code=True)
print(f"Tipo:          {type(ds).__name__}")            # Dataset
print(f"Número filas: {len(ds)}")                       # debe ser 500
print(f"Columnas:      {ds.column_names}")              # nombres de columnas del dataset
print(f"Features:\n{ds.features}")                       # tipos HF (Value, Sequence...)

## Bloque 2 — Mirar un documento entero

Antes de decidir el schema, hay que mirar los datos. **Siempre.** Truncamos el texto al imprimir porque suele ser muy largo.

In [ ]:
doc = ds[0]                                             # primera fila como dict Python
for k, v in doc.items():
    if isinstance(v, str) and len(v) > 300:
        print(f"\n--- {k} (string, {len(v)} chars, truncado) ---")
        print(v[:300] + " ...")
    elif isinstance(v, list) and len(v) > 10:
        print(f"\n--- {k} (list, {len(v)} items, truncado) ---")
        print(v[:10], "...")
    else:
        print(f"\n--- {k} ---")
        print(v)

**Lo que vas a ver (esperado):**\n- `celex_id`: string como `\"31979D0509\"` — ID oficial EUR-Lex.\n- `title`: string corto con el título.\n- `text`: string largo con el cuerpo completo (header + recitals + articulado).\n- `eurovoc_concepts`: lista de strings que parecen números (`[\"192\", \"2356\", ...]`). **Aunque parezcan ints, vienen como strings** — el publisher EU los trata como identificadores opacos, no como enteros para hacer aritmética. Nuestro schema tiene que respetarlo: `ArrayType(StringType())`, no `ArrayType(LongType())`.\n\nEn silver enriqueceremos parsando el `celex_id` (los primeros 4 dígitos tras la letra inicial son el año: `31979D0509` → año 1979, `D` = Decision).

## Bloque 3 — Convertir a Spark DataFrame con schema explícito

**Decisión de diseño:** `Dataset` HF → Spark DataFrame. Dos opciones:

1. `Dataset.to_pandas()` → `spark.createDataFrame(pdf)`. Cómodo, pero carga todo en memoria del driver (NO escala más allá de unos GB).
2. `Dataset.to_parquet(tmpfile)` → `spark.read.parquet(tmpfile)`. Más pasos pero escala bien y respeta la lazy evaluation de Spark.

**Para 500 filas en exploración** → opción 1.  **Para 55k+ en producción** → opción 2 (lo veremos en `spark/ingestion.py`).

**Schema explícito:** lo definimos *después de ver los datos*. Forzar `LongType` para `celex_id` sería un error si los IDs llevan letras — lo dejamos como `StringType`.

In [ ]:
# Schema bronze: mínimo y fiel a lo que viene de HF. Aquí NO transformamos
# nada — ése es el principio de bronze. Los nombres y tipos respetan exactamente
# lo que devuelve HF (ver Bloque 1).
schema_bronze = StructType([
    StructField("celex_id",         StringType(),              nullable=False),  # ID oficial EUR-Lex
    StructField("title",            StringType(),              nullable=True),    # título legible
    StructField("text",             StringType(),              nullable=False),  # cuerpo completo
    StructField("eurovoc_concepts", ArrayType(StringType()),   nullable=True),    # códigos EuroVoc como strings
])
schema_bronze

In [ ]:
# pandas en memoria → preparamos para pasar a Spark con el schema ya definido.
# Reordenamos las columnas para que casen con el orden del StructType (Spark
# es estricto con el orden al construir desde pandas con schema explícito).
pdf = ds.to_pandas()                                    # 500 filas → OK en RAM
pdf = pdf[["celex_id", "title", "text", "eurovoc_concepts"]]
print(f"dtypes pandas:\n{pdf.dtypes}\n")
print(f"primeras filas (sin texto):")
pdf[["celex_id", "title", "eurovoc_concepts"]].head()

In [ ]:
# pandas → Spark con schema explícito (forzamos los tipos, no inferimos)
df_bronze = spark.createDataFrame(pdf, schema=schema_bronze)
df_bronze.printSchema()
print(f"Número de filas: {df_bronze.count()}")
print(f"Particiones:     {df_bronze.rdd.getNumPartitions()}")
(
    df_bronze
    .select(
        "celex_id",
        F.length("title").alias("title_len"),
        F.length("text").alias("text_len"),
        F.size("eurovoc_concepts").alias("n_concepts"),
    )
    .show(5)
)

## Bloque 4 — Escribir a `data/bronze/eurlex/` como Parquet

**¿Por qué Parquet y no JSON/CSV?**
- **Columnar**: cuando en silver leas sólo `celex_id`, Spark salta el `text` (que es la mayor parte del disco). Predicate y projection pushdown gratis.
- **Comprimido**: snappy por defecto, suele bajar 3-5× el tamaño frente a JSON.
- **Tipado**: el schema viaja con el fichero. Al volver a leer no hay que volver a inferir.

**Convención de carpetas:** `data/bronze/<source>/<dataset>/` para que un día podamos tener `data/bronze/eurlex/`, `data/bronze/eurlex_sparql/`, etc.

**Modo `overwrite`**: bronze es idempotente — si re-ejecutas, sobrescribes. Es el principio: cualquier capa se reconstruye desde cero desde la anterior. Si quisiera *append* tendría que gestionar deduplicación manualmente.

In [ ]:
# Ruta absoluta al volumen montado del host (mismo patrón que en Week 2)
bronze_path = Path("/home/jovyan/work/data/bronze/eurlex")
bronze_path.mkdir(parents=True, exist_ok=True)

(
    df_bronze.write
    .mode("overwrite")                                  # bronze es reproducible
    .parquet(str(bronze_path))                          # escribe partition files
)

# Listar los ficheros creados
for p in sorted(bronze_path.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:60s} {size_kb:>8.1f} KB")

Observa los ficheros: `part-00000-*.parquet`, `part-00001-*.parquet`... Hay tantos como **particiones** tenía el DataFrame al escribir. `_SUCCESS` es un marker vacío que indica que el job terminó bien (lo usa Hadoop/Hive para distinguir directorios completos de jobs interrumpidos).

## Bloque 5 — Sanity check: leer de vuelta

Round-trip: escribimos → leemos → verificamos que cuadran las filas y el schema.

In [ ]:
df_check = spark.read.parquet(str(bronze_path))
df_check.printSchema()
print(f"Filas leídas: {df_check.count()}")
# Top-5 documentos por número de eurovoc concepts (verifica que arrays se preservan)
(
    df_check
    .select("celex_id", F.size("eurovoc_concepts").alias("n_concepts"))
    .orderBy(F.desc("n_concepts"))
    .show(5)
)

---
## Bronze cerrado ✓

La lógica de bronze ya está extraída a `spark/schemas.py` + `spark/ingestion.py` (función reusable y testeada en `tests/test_ingestion.py`, ejecutable con `make ingest-bronze`).

Ahora vamos con **Silver** 👇

---
# Parte 2 — Bronze → Silver

**Silver = limpio, estructurado, con la lógica de dominio EU.** Tres bloques:
1. **Parsear `celex_id`** → derivar `sector`, `year`, `doc_type` y `doc_type_label`.
2. **Limpiar texto** → quitar la cabecera del Diario Oficial; preservar párrafos (los necesita el chunking de RAG en la fase 3).
3. **Particionar** por `year` y `doc_type` (Hive-style).

La lógica vive en `spark/transformations.py` y `spark/quality_checks.py` (testeada en `tests/test_transformations.py`). Aquí la **importamos y la vemos en acción** — los módulos se reutilizan tal cual desde el notebook.

## Demo: Sort vs Exchange (la deuda de Week 2)

En Week 2 vimos *narrow* vs *wide* transformations pero el job de bronze no tenía shuffle. Ahora sí: un `groupBy` es **wide** → dispara un **Exchange** (shuffle: los datos cruzan la red para juntar las mismas claves).

👉 **Abre http://localhost:4040 ANTES de ejecutar la siguiente celda.** En la pestaña *SQL / DataFrame* verás el plan con un nodo `Exchange`; en *Stages* verás cómo el job se parte en 2 stages (antes y después del shuffle).

In [ ]:
# El kernel de Jupyter tiene su CWD en notebooks/, no en la raíz del repo, así
# que el paquete spark/ no está en sys.path. Lo añadimos para poder importar los
# módulos (pytest no lo necesita porque el Makefile hace 'cd' a la raíz antes).
import sys

REPO_ROOT = "/home/jovyan/work"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Importamos la lógica silver desde los módulos (mismo código que testea pytest).
from spark.transformations import (
    with_celex_components,
    with_doc_type_label,
    with_clean_text,
)
from spark.quality_checks import run_silver_checks

bronze = spark.read.parquet("/home/jovyan/work/data/bronze/eurlex")
print("Bronze rows:", bronze.count())

# DEMO SHUFFLE: groupBy = transformación WIDE -> Exchange.
demo = (
    bronze
    .withColumn("doc_type", F.regexp_extract("celex_id", r"^\d\d{4}([A-Z])", 1))
    .groupBy("doc_type").count()
)
demo.explain(mode="formatted")   # busca el nodo 'Exchange hashpartitioning'
demo.show()

**Lo que acabas de ver en el plan:**

| Tipo | Operaciones | ¿Shuffle? | Coste |
|------|-------------|-----------|-------|
| **Narrow** | `select`, `filter`, `withColumn`, `regexp_extract` | No | barato: cada partición se procesa sola |
| **Wide** | `groupBy`, `join`, `distinct`, `repartition` | **Sí (Exchange)** | caro: datos cruzan la red |

El `Exchange hashpartitioning(doc_type, ...)` reparte las filas por hash de `doc_type` para que todas las "R" caigan en la misma tarea y se cuenten juntas.

**Patrón transferible:** este "junta-por-clave-antes-de-agregar" es el `GROUP BY` de SQL (Week 1) y reaparecerá como dependencias entre tareas en Airflow (Week 4). Reconocer dónde hay un shuffle = saber dónde está el cuello de botella de un job Spark.

In [ ]:
# Encadenamos las 3 transformaciones con .transform() (composición tipo pipe).
# Catalyst las fusiona en pocos 'Project' (lazy eval): escribes pasos legibles,
# el optimizador los colapsa sin materializar intermedios.
silver = (
    bronze
    .transform(with_celex_components)
    .transform(with_doc_type_label)
    .transform(with_clean_text)
    .select("celex_id", "sector", "year", "doc_type",
            "doc_type_label", "title", "text", "eurovoc_concepts")
)
silver.select("celex_id", "year", "doc_type", "doc_type_label").show(8)

# Antes/después de limpiar: la cabecera del Diario Oficial desaparece.
cid = "32014R0727"
print("=== BRONZE text[:120] ===")
print(repr(bronze.filter(F.col("celex_id") == cid).collect()[0].text[:120]))
print("\n=== SILVER text[:120] ===")
print(repr(silver.filter(F.col("celex_id") == cid).collect()[0].text[:120]))

In [ ]:
from pathlib import Path
silver_path = Path("/home/jovyan/work/data/silver/eurlex")

# repartition("year","doc_type") ANTES de partitionBy: shuffle deliberado que
# coloca cada (year, doc_type) en una sola tarea -> 1 fichero por carpeta.
# Sin él, cada partición de entrada escribiría en cada carpeta -> small files.
(
    silver
    .repartition("year", "doc_type")
    .write.mode("overwrite")
    .partitionBy("year", "doc_type")
    .parquet(str(silver_path))
)

n_dirs  = len(list(silver_path.glob("year=*/doc_type=*")))
n_files = len(list(silver_path.glob("year=*/doc_type=*/*.parquet")))
print(f"Carpetas de partición: {n_dirs} | ficheros .parquet: {n_files}")

## ⚠️ Lección: el "small files problem"

Con ~100 filas hemos generado ~50 carpetas de partición (`year=YYYY/doc_type=X`), una por combinación, con **1 fichero diminuto cada una** (~2 filas/fichero). El `repartition` evitó que fuera *peor* (1 fichero por carpeta en vez de N×M), pero el **esquema de particionado es demasiado fino para este volumen**.

**Por qué importa:** miles de ficheros pequeños matan a Spark — cada fichero es una tarea, un open/close de disco, una entrada de metadatos. La regla práctica es apuntar a ficheros de **~128 MB–1 GB**.

**Trade-off de cardinalidad:** particionar acelera queries que filtran por la columna de partición (*partition pruning*: `WHERE year=2014` lee solo esa carpeta), pero `year` (~40 valores) × `doc_type` (3) sobre-particiona con pocos datos. En producción con 57k+ docs es más razonable; aun así `year` puede ser demasiado fino y a veces se agrupa por **década**. Lo dejamos por `year`+`doc_type` porque demuestra el patrón Hive y es lo que el plan pide.

In [ ]:
# Quality gate: releemos silver y validamos el contrato antes de pasar a gold.
# En Airflow (Week 4) esto sería una tarea-gate: si falla, el DAG para aquí.
silver_back = spark.read.parquet(str(silver_path))
report = run_silver_checks(silver_back)
print("rows:", report.total_rows, "| is_ok:", report.is_ok,
      "| violations:", report.total_violations)
print("nulls:", report.null_violations,
      "| short_text:", report.short_text_rows,
      "| year_oor:", report.year_out_of_range_rows)

silver_back.groupBy("doc_type", "doc_type_label").count() \
    .orderBy(F.desc("count")).show()

In [ ]:
spark.stop()